# Hugging Face 모델 로컬 실행


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


작은 instruct 모델을 `HuggingFacePipeline`으로 로드한 뒤 `ChatHuggingFace`로 감싸 채팅 템플릿을 적용합니다. 최초 실행에는 모델 다운로드가 필요하며 CPU에서는 시간이 걸릴 수 있습니다.


In [ ]:
%pip install -qU langchain-huggingface transformers torch accelerate python-dotenv


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
cache_dir = Path("cache/huggingface")
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(cache_dir.resolve()))


In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

model_id = os.getenv("HF_LOCAL_MODEL", "HuggingFaceTB/SmolLM2-1.7B-Instruct")
pipeline_llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "repetition_penalty": 1.03,
        "return_full_text": False,
    },
)
chat = ChatHuggingFace(llm=pipeline_llm)

response = chat.invoke(
    [
        ("system", "간결한 한국어로 답하세요."),
        ("human", "RAG를 한 문장으로 설명해 주세요."),
    ]
)
print(response.text)


## LCEL 요약 체인


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "입력 글을 중요한 순서대로 세 개의 불릿으로 요약하세요."),
        ("human", "{text}"),
    ]
)
chain = prompt | chat | StrOutputParser()

text = (
    "대규모 언어 모델은 많은 텍스트로 학습되어 다음 토큰을 예측합니다. "
    "다양한 작업을 수행하지만 사실 오류와 편향이 생길 수 있습니다. "
    "검색 증강 생성은 외부 근거를 제공해 답변의 최신성과 검증 가능성을 높입니다."
)
print(chain.invoke({"text": text}))


메모리가 부족하면 더 작은 모델을 선택하거나 별도의 양자화 구성을 사용하세요. `device`와 `device_map`을 동시에 지정하지 않으며, 양자화 옵션은 설치된 PyTorch·CUDA·bitsandbytes 조합에 맞춰 별도로 검증합니다.
